# Velocity preprocessing: export Seurat data

Remote source reviewed from `20250509-doublet-p38/2-anno/velocity/velocity.ipynb`. Run from top to bottom after configuring `config/paths.*`.


In [ ]:
# Centralized paths and scheduler-aware thread limits.
repo_root <- if (file.exists("config/paths.R")) "." else if (file.exists("../config/paths.R")) ".." else stop("Run Jupyter from the repository root or notebooks/ directory.")
source(file.path(repo_root, "config", "paths.R"))


In [1]:
library(Seurat)
library(Matrix)

out_dir <- legacy_path("20250509-doublet-p38/2-anno/velocity")
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

# 加载Seurat对象
combined <- readRDS(legacy_path("20250509-doublet-p38/1QC/2stsub.rds"))
combined$seurat_clusters <- combined$RNA_snn_res.1.2
DefaultAssay(combined) <- "RNA"
rownames(combined) <- make.unique(rownames(combined))

# 合并layers (如果使用的是Seurat v5)
# 如果是Seurat v4或更早版本，并且没有分割的layers，这一步可能不需要或会报错
# 请根据您的Seurat版本和对象结构确认是否需要 JoinLayers()
# 假设您的Seurat对象已经是标准的Assay结构，如果没有layers需要join，可以注释掉下一行
# 如果您是通过例如 Read10X -> CreateSeuratObject -> merge 创建的，可能不需要JoinLayers
# 如果您是用例如 SCTransform 后再 merge，或者有其他方式创建了分层assay，则可能需要
# 鉴于您之前的脚本中有这一行，我暂时保留。如果报错，请检查您的对象结构。
if ("layers" %in% slotNames(combined@assays$RNA)) {
  combined <- JoinLayers(combined) # 仅当RNA assay有layers时执行
}


# 查看样本信息
print("原始样本信息:")
print(table(combined$orig.ident))

# 添加样本前缀到barcode - 优化逻辑
current_barcodes <- colnames(combined)
new_barcodes <- character(length(current_barcodes)) # 初始化新barcode向量

wt_prefix_str <- "WT_"
ko_prefix_str <- "KO_"

for (i in seq_along(current_barcodes)) {
  barcode_val <- current_barcodes[i]
  ident_val <- combined$orig.ident[i]

  # 1. 移除可能已存在的旧项目前缀 (WT_ 或 KO_)，得到核心barcode
  # 这对于处理可能已经带有项目级别前缀（如来自不同10X运行的cellranger id）的barcode很重要
  # 例如，如果barcode是 "sample1_AAACCCAAGACTTCAC-1"，我们希望保留 "AAACCCAAGACTTCAC-1"
  # 但在这里，我们的目标是确保我们的WT/KO前缀是唯一且正确的
  
  core_barcode <- barcode_val
  # 移除已存在的WT_或KO_前缀，以防万一
  if (startsWith(core_barcode, wt_prefix_str)) {
    core_barcode <- sub(paste0("^", wt_prefix_str), "", core_barcode)
  }
  if (startsWith(core_barcode, ko_prefix_str)) {
    core_barcode <- sub(paste0("^", ko_prefix_str), "", core_barcode)
  }
  
  # 2. 根据orig.ident添加新的、正确的前缀
  if (ident_val == "WT") {
    new_barcodes[i] <- paste0(wt_prefix_str, core_barcode)
  } else if (ident_val == "KO") {
    new_barcodes[i] <- paste0(ko_prefix_str, core_barcode)
  } else {
    warning(paste("未知的orig.ident:", ident_val, "对于barcode:", barcode_val, "将保留原始barcode。"))
    new_barcodes[i] <- barcode_val # 保留原始barcode以防有其他样本类型
  }
}

# 更新Seurat对象的barcode
# 首先检查是否有重复的新barcode，这通常不应该发生，除非原始barcode在去除前缀后有重复
if (any(duplicated(new_barcodes))) {
  warning("生成的新barcode中存在重复值！请检查您的orig.ident和原始barcode。")
  # 可以考虑在这里添加make.unique(new_barcodes)如果确实需要，但这通常表明上游有问题
}
renamed_combined <- RenameCells(combined, new.names = new_barcodes)

# 打印新barcode示例，用于调试
print("添加前缀后的barcode样例 (WT):")
print(head(colnames(renamed_combined)[renamed_combined$orig.ident == "WT"]))
print("添加前缀后的barcode样例 (KO):")
print(head(colnames(renamed_combined)[renamed_combined$orig.ident == "KO"]))


# 导出表达矩阵
# Seurat v5 使用 LayerData, v3/v4 使用 GetAssayData(layer=)
# 为了兼容性，我们先检查
if ("counts" %in% Layers(renamed_combined[["RNA"]])) {
    counts_data <- LayerData(renamed_combined, assay = "RNA", layer = "counts")
} else {
    counts_data <- GetAssayData(renamed_combined, assay = "RNA", slot = "counts") # for older Seurat or if not using layers
}
writeMM(counts_data, file = file.path(out_dir, "matrix_counts.mtx"))

if ("data" %in% Layers(renamed_combined[["RNA"]])) {
    norm_data <- LayerData(renamed_combined, assay = "RNA", layer = "data")
} else {
    norm_data <- GetAssayData(renamed_combined, assay = "RNA", slot = "data") # for older Seurat or if not using layers
}
writeMM(norm_data, file = file.path(out_dir, "matrix_normalized.mtx"))


# 导出基因和细胞信息 - 这里使用新的barcode
genes <- data.frame(gene_name = rownames(renamed_combined[["RNA"]])) # 确保从assay获取
write.csv(genes, file = file.path(out_dir, "genes.csv"), row.names = FALSE, quote = FALSE)

cells <- data.frame(barcode = colnames(renamed_combined))
write.csv(cells, file = file.path(out_dir, "barcodes.csv"), row.names = FALSE, quote = FALSE)

# 导出元数据
meta <- renamed_combined@meta.data
# 确保元数据的行名与导出的barcodes.csv中的barcode一致
# RenameCells已经更新了meta.data的行名
write.csv(meta, file = file.path(out_dir, "metadata.csv"), quote = FALSE)

# 导出降维结果
if ("pca" %in% names(renamed_combined@reductions)) {
  pca_coords <- Embeddings(renamed_combined[["pca"]])
  write.csv(pca_coords, file = file.path(out_dir, "pca.csv"), quote = FALSE)
  
  if (!is.null(renamed_combined[["pca"]]@stdev)) {
    stdev <- data.frame(stdev = renamed_combined[["pca"]]@stdev)
    write.csv(stdev, file = file.path(out_dir, "pca_stdev.csv"), row.names = FALSE, quote = FALSE)
  }
} else {
  print("PCA降维结果未找到。")
}

if ("umap" %in% names(renamed_combined@reductions)) {
  umap_coords <- Embeddings(renamed_combined[["umap"]])
  write.csv(umap_coords, file = file.path(out_dir, "umap.csv"), quote = FALSE)
} else {
  print("UMAP降维结果未找到。")
}

print(paste("数据导出完成，已保存到目录:", out_dir))

Loading required package: SeuratObject

Loading required package: sp

‘SeuratObject’ was built under R 4.4.1 but the current version is
4.4.2; it is recomended that you reinstall ‘SeuratObject’ as the ABI
for R may have changed


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t




[1] "原始样本信息:"

  KO   WT 
8982 9565 
[1] "添加前缀后的barcode样例 (WT):"
[1] "WT_AAACCCAAGATGCAGC-1" "WT_AAACCCAAGCAACAGC-1" "WT_AAACCCAAGTAGGATT-1"
[4] "WT_AAACCCAAGTTCGGTT-1" "WT_AAACCCACATCTTTCA-1" "WT_AAACCCAGTACTGTTG-1"
[1] "添加前缀后的barcode样例 (KO):"
[1] "KO_AAACCCAAGACTTCAC-1" "KO_AAACCCAAGGACAGCT-1" "KO_AAACCCAAGGCCTAAG-1"
[4] "KO_AAACCCAGTATGTGTC-1" "KO_AAACCCATCCTAGAGT-1" "KO_AAACCCATCTAGACAC-1"


NULL

NULL

[1] "数据导出完成，已保存到目录: /data3/Group8/gonglihao/20250509-doublet-p38/2-anno/velocity"
